In [ ]:
#!/usr/bin/env python3
# Object Detection (concise, correct Content-Type). Requires: pillow, requests, config.py (HF_API_KEY).
import os, io, time, random, requests, mimetypes
from datetime import datetime
from PIL import Image, ImageDraw, ImageFont
from config import HF_API_KEY

MODEL = "facebook/detr-resnet-101"
API = f"https://router.huggingface.co/hf-inference/models/{MODEL}"
ALLOWED, MAX_MB = {".jpg",".jpeg",".png",".bmp",".gif",".webp",".tiff"}, 8
EMOJI = {"person":"🧍","car":"🚗","truck":"🚚","bus":"🚌","bicycle":"🚲","motorcycle":"🏍️","dog":"🐶","cat":"🐱",
         "bird":"🐦","horse":"🐴","sheep":"🐑","cow":"🐮","bear":"🐻","giraffe":"🦒","zebra":"🦓","banana":"🍌",
         "apple":"🍎","orange":"🍊","pizza":"🍕","broccoli":"🥦","book":"📘","laptop":"💻","tv":"📺","bottle":"🧴","cup":"🥤"}

def font(sz=18):
    for f in ("DejaVuSans.ttf","arial.ttf"):
        try: return ImageFont.truetype(f, sz)
        except: pass
    return ImageFont.load_default()

def ask_image():
    print("\n🎯 Pick an image (JPG/PNG/WebP/BMP/TIFF ≤ 8MB) from this folder.")
    while True:
        p = input("Image path: ").strip().strip('"').strip("'")
        if not p or not os.path.isfile(p): print("⚠️ Not found."); continue
        if os.path.splitext(p)[1].lower() not in ALLOWED: print("⚠️ Unsupported type."); continue
        if os.path.getsize(p)/(1024*1024) > MAX_MB: print("⚠️ Too big (>8MB)."); continue
        try: Image.open(p).verify()
        except: print("⚠️ Corrupted image."); continue
        return p

def infer(path, img_bytes, tries=8):
    mime, _ = mimetypes.guess_type(path)
    # Prefer exact image/* content-type; fall back to multipart if guess fails
    for _ in range(tries):
        if mime and mime.startswith("image/"):
            r = requests.post(API,
                headers={"Authorization": f"Bearer {HF_API_KEY}",
                         "Content-Type": mime},
                data=img_bytes, timeout=60)
        else:
            r = requests.post(API,
                headers={"Authorization": f"Bearer {HF_API_KEY}"},
                files={"inputs": (os.path.basename(path), img_bytes, "application/octet-stream")},
                timeout=60)
        if r.status_code == 200:
            d = r.json()
            if isinstance(d, dict) and "error" in d: raise RuntimeError(d["error"])
            if not isinstance(d, list): raise RuntimeError("Bad API response.")
            return d
        if r.status_code == 503: time.sleep(2); continue
        raise RuntimeError(f"API {r.status_code}: {r.text[:300]}")
    raise RuntimeError("Model warm-up timeout.")

def draw(img, dets, thr=0.5):
    d = ImageDraw.Draw(img); f = font(18); counts = {}
    for det in dets[:50]:
        s = float(det.get("score",0));
        if s < thr: continue
        lab = det.get("label","object"); b = det.get("box",{})
        x1,y1,x2,y2 = (int(b.get(k,0)) for k in ("xmin","ymin","xmax","ymax"))
        if not (x2>0 and y2>0):
            x,y,w,h = int(b.get("x",0)), int(b.get("y",0)), int(b.get("w",0)), int(b.get("h",0))
            x1,y1,x2,y2 = x,y,x+w,y+h
        color = tuple(random.randint(80,255) for _ in range(3))
        d.rectangle([(x1,y1),(x2,y2)], outline=color, width=4)
        txt = f"{EMOJI.get(lab.lower(),'✨')} {lab} {s*100:.0f}%"
        tw,th = d.textlength(txt,font=f), f.size+6
        d.rectangle([(x1,max(0,y1-th)),(x1+tw+8,y1)], fill=color)
        d.text((x1+4,y1-th+3), txt, font=f, fill=(0,0,0))
        counts[lab] = counts.get(lab,0)+1
    return counts

def main():
    path = ask_image()
    with open(path,"rb") as fh: by = fh.read()
    try: dets = infer(path, by)
    except Exception as e: return print("❌", e)
    img = Image.open(io.BytesIO(by)).convert("RGB")
    counts = draw(img, dets, thr=0.5)
    out = f"annotated_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
    img.save(out); print(f"✅ Saved: {out}")
    if counts:
        print("🎉 I found:");
        for k,v in sorted(counts.items(), key=lambda kv:(-kv[1],kv[0])):
            print(f" • {EMOJI.get(k.lower(),'✨')} {k}: {v}")
    else:
        print("🤔 No confident detections—try a clearer or busier scene!")

    print("\n⚠️ Disclaimer: This is an AI model demo. "
          "Detections may not always be accurate or complete. "
          "Use it for fun and learning, not for safety-critical decisions.")

if __name__=="__main__": main()


```python
#!/usr/bin/env python3
# This is a shebang line, indicating that the script should be executed with python3. It's often used in executable scripts but less critical in a notebook.

# Object Detection (concise, correct Content-Type). Requires: pillow, requests, config.py (HF_API_KEY).
# This comment describes the purpose of the script (Object Detection) and lists its dependencies.

import os, io, time, random, requests, mimetypes
# Imports several standard Python modules:
# os: Provides functions for interacting with the operating system (e.g., file paths).
# io: Provides tools for working with I/O streams (e.g., reading/writing files).
# time: Provides various time-related functions (e.g., for pauses).
# random: Generates pseudo-random numbers (used for colors).
# requests: Allows sending HTTP requests (to the Hugging Face API).
# mimetypes: Guesses the MIME type of a file.

from datetime import datetime
# Imports the datetime class from the datetime module for working with dates and times (used for output filename).

from PIL import Image, ImageDraw, ImageFont
# Imports Image, ImageDraw, and ImageFont classes from the Pillow (PIL) library for image manipulation (opening, drawing, fonts).

from config import HF_API_KEY
# Imports the HF_API_KEY variable from a local 'config.py' file, which is assumed to contain the Hugging Face API key.

MODEL = "facebook/detr-resnet-101"
# Defines the specific object detection model to be used from Hugging Face.

API = f"https://router.huggingface.co/hf-inference/models/{MODEL}"
# Constructs the full API endpoint URL for the chosen model using an f-string.

ALLOWED, MAX_MB = {".jpg",".jpeg",".png",".bmp",".gif",".webp",".tiff"}, 8
# Defines a set of allowed image file extensions and the maximum allowed file size in megabytes (8MB).

EMOJI = {"person":"🧍","car":"🚗","truck":"🚚","bus":"🚌","bicycle":"🚲","motorcycle":"🏍️","dog":"🐶","cat":"🐱",
         "bird":"🐦","horse":"🐴","sheep":"🐑","cow":"🐮","bear":"🐻","giraffe":"🦒","zebra":"🦓","banana":"🍌",
         "apple":"🍎","orange":"🍊","pizza":"🍕","broccoli":"🥦","book":"📘","laptop":"💻","tv":"📺","bottle":"🧴","cup":"🥤"}
# A dictionary mapping common object labels to their corresponding emojis for better visualization.

def font(sz=18):
    # Defines a function to load a suitable font for drawing text on images.
    for f in ("DejaVuSans.ttf","arial.ttf"):
        # Tries to load specific fonts ('DejaVuSans.ttf', 'arial.ttf') at the given size.
        try: return ImageFont.truetype(f, sz)
        except: pass
        # If a font fails to load (e.g., not found), it catches the exception and tries the next one.
    return ImageFont.load_default()
    # If neither custom font is found, it falls back to Pillow's default font.

def ask_image():
    # Defines a function to prompt the user for an image path and validate it.
    print("\n🎯 Pick an image (JPG/PNG/WebP/BMP/TIFF ≤ 8MB) from this folder.")
    # Prints instructions for the user on how to select an image.
    while True:
        # Enters an infinite loop that continues until a valid image path is provided.
        p = input("Image path: ").strip().strip('"').strip("'")
        # Prompts the user for an image path, then removes leading/trailing whitespace, quotes.
        if not p or not os.path.isfile(p): print("⚠️ Not found."); continue
        # Checks if the path is empty or if a file doesn't exist at the given path; if so, prints an error and continues the loop.
        if os.path.splitext(p)[1].lower() not in ALLOWED: print("⚠️ Unsupported type."); continue
        # Extracts the file extension, converts to lowercase, and checks if it's in the ALLOWED set; if not, prints an error and continues.
        if os.path.getsize(p)/(1024*1024) > MAX_MB: print("⚠️ Too big (>8MB)."); continue
        # Gets the file size in bytes, converts to MB, and checks if it exceeds MAX_MB; if so, prints an error and continues.
        try: Image.open(p).verify()
        # Attempts to open the image using Pillow and verifies its integrity.
        except: print("⚠️ Corrupted image."); continue
        # If the image is corrupted, it catches the exception, prints an error, and continues the loop.
        return p
        # If all checks pass, the valid image path is returned, exiting the function and the loop.

def infer(path, img_bytes, tries=8):
    # Defines a function to send an image to the Hugging Face API for object detection inference.
    mime, _ = mimetypes.guess_type(path)
    # Guesses the MIME type of the image based on its file path.
    # Prefer exact image/* content-type; fall back to multipart if guess fails
    # A comment explaining the content-type strategy for the API request.
    for _ in range(tries):
        # Loops for a specified number of tries to handle potential API issues (like model warm-up or temporary errors).
        if mime and mime.startswith("image/"):
            # If a specific image MIME type is successfully determined...
            r = requests.post(API,
                headers={"Authorization": f"Bearer {HF_API_KEY}",
                         "Content-Type": mime},
                data=img_bytes, timeout=60)
            # ...send a POST request with the image bytes directly and the specific MIME type header.
        else:
            # If the MIME type is not specific or could not be guessed...
            r = requests.post(API,
                headers={"Authorization": f"Bearer {HF_API_KEY}"},
                files={"inputs": (os.path.basename(path), img_bytes, "application/octet-stream")},
                timeout=60)
            # ...send a POST request using multipart form data with a generic content-type.
        if r.status_code == 200:
            # If the API request is successful (HTTP 200 OK).
            d = r.json()
            # Parses the JSON response from the API.
            if isinstance(d, dict) and "error" in d: raise RuntimeError(d["error"])
            # Checks if the JSON response indicates an error; if so, raises a RuntimeError.
            if not isinstance(d, list): raise RuntimeError("Bad API response.")
            # Ensures the response is a list (expected format for detections); otherwise, raises an error.
            return d
            # Returns the list of detection results.
        if r.status_code == 503: time.sleep(2); continue
        # If the service is temporarily unavailable (HTTP 503), waits for 2 seconds and retries the request.
        raise RuntimeError(f"API {r.status_code}: {r.text[:300]}")
        # For any other non-200, non-503 status code, raises a RuntimeError with details.
    raise RuntimeError("Model warm-up timeout.")
    # If all retries fail without a successful response, raises a timeout error.

def draw(img, dets, thr=0.5):
    # Defines a function to draw bounding boxes and labels on an image based on detection results.
    d = ImageDraw.Draw(img); f = font(18); counts = {}
    # Initializes an ImageDraw object for drawing, loads a font, and creates an empty dictionary to count detected objects.
    for det in dets[:50]:
        # Iterates through the first 50 detection results (to avoid cluttering for many detections).
        s = float(det.get("score",0));
        # Retrieves the confidence score of the detection; defaults to 0 if not present.
        if s < thr: continue
        # If the confidence score is below the threshold (thr), skips to the next detection.
        lab = det.get("label","object"); b = det.get("box",{})
        # Retrieves the object label and its bounding box dictionary; defaults to 'object' and an empty dict.
        x1,y1,x2,y2 = (int(b.get(k,0)) for k in ("xmin","ymin","xmax","ymax"))
        # Extracts the bounding box coordinates (xmin, ymin, xmax, ymax) as integers.
        if not (x2>0 and y2>0):
            # Checks if the bounding box coordinates are valid (non-zero width/height).
            x,y,w,h = int(b.get("x",0)), int(b.get("y",0)), int(b.get("w",0)), int(b.get("h",0))
            # If not, assumes a different format (x, y, width, height) and extracts those values.
            x1,y1,x2,y2 = x,y,x+w,y+h
            # Converts the (x, y, width, height) format to (xmin, ymin, xmax, ymax).
        color = tuple(random.randint(80,255) for _ in range(3))
        # Generates a random RGB color for the bounding box and label background.
        d.rectangle([(x1,y1),(x2,y2)], outline=color, width=4)
        # Draws the bounding box rectangle on the image with the generated color and a width of 4 pixels.
        txt = f"{EMOJI.get(lab.lower(),'✨')} {lab} {s*100:.0f}%"
        # Formats the text label to include an emoji (if available), the object label, and its confidence score as a percentage.
        tw,th = d.textlength(txt,font=f), f.size+6
        # Calculates the width and height required for the text label using the loaded font.
        d.rectangle([(x1,max(0,y1-th)),(x1+tw+8,y1)], fill=color)
        # Draws a filled rectangle behind the text label, positioned just above the bounding box, using the same color.
        d.text((x1+4,y1-th+3), txt, font=f, fill=(0,0,0))
        # Draws the text label on top of the background rectangle with a black color.
        counts[lab] = counts.get(lab,0)+1
        # Increments the count for the detected object label in the 'counts' dictionary.
    return counts
    # Returns the dictionary containing counts of each detected object.

def main():
    # The main function that orchestrates the entire object detection workflow.
    path = ask_image()
    # Calls the ask_image() function to get a validated path to an input image from the user.
    with open(path,"rb") as fh: by = fh.read()
    # Opens the image file in binary read mode ('rb') and reads its entire content into 'by'.
    try: dets = infer(path, by)
    # Attempts to perform object detection inference by calling the infer() function with the image path and bytes.
    except Exception as e: return print("❌", e)
    # If any exception occurs during inference, it catches the error, prints an error message, and exits the function.
    img = Image.open(io.BytesIO(by)).convert("RGB")
    # Opens the image bytes in memory as a Pillow Image object and converts it to 'RGB' format.
    counts = draw(img, dets, thr=0.5)
    # Calls the draw() function to annotate the image with the detection results and gets the counts of detected objects.
    out = f"annotated_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
    # Generates a unique output filename for the annotated image using the current timestamp.
    img.save(out); print(f"✅ Saved: {out}")
    # Saves the annotated image to the generated filename and prints a confirmation message.
    if counts:
        # Checks if any objects were confidently detected (i.e., if 'counts' is not empty).
        print("🎉 I found:");
        # Prints a celebratory message.
        for k,v in sorted(counts.items(), key=lambda kv:(-kv[1],kv[0])):
            # Iterates through the detected objects, sorted by count (descending) and then by name (ascending).
            print(f" • {EMOJI.get(k.lower(),'✨')} {k}: {v}")
            # Prints each detected object, its emoji, name, and count.
    else:
        # If no objects were confidently detected.
        print("🤔 No confident detections—try a clearer or busier scene!")
        # Prints a message indicating no confident detections and suggests improvements.

    print("\n⚠️ Disclaimer: This is an AI model demo. "
          "Detections may not always be accurate or complete. "
          "Use it for fun and learning, not for safety-critical decisions.")
    # Prints a disclaimer about the nature of AI model demos.

if __name__=="__main__": main()
# This conditional ensures that the main() function is called only when the script is executed directly (not when imported as a module).
```

In [ ]:
#!/usr/bin/env python3
# Object Detection (concise, correct Content-Type). Requires: pillow, requests, config.py (HF_API_KEY).
import os, io, time, random, requests, mimetypes
from datetime import datetime
from PIL import Image, ImageDraw, ImageFont
from config import HF_API_KEY

MODEL = "facebook/detr-resnet-101"
API = f"https://router.huggingface.co/hf-inference/models/{MODEL}"
ALLOWED, MAX_MB = {".jpg",".jpeg",".png",".bmp",".gif",".webp",".tiff"}, 8
EMOJI = {"person":"🧍","car":"🚗","truck":"🚚","bus":"🚌","bicycle":"🚲","motorcycle":"🏍️","dog":"🐶","cat":"🐱",
         "bird":"🐦","horse":"🐴","sheep":"🐑","cow":"🐮","bear":"🐻","giraffe":"🦒","zebra":"🦓","banana":"🍌",
         "apple":"🍎","orange":"🍊","pizza":"🍕","broccoli":"🥦","book":"📘","laptop":"💻","tv":"📺","bottle":"🧴","cup":"🥤"}

def font(sz=18):
    for f in ("DejaVuSans.ttf","arial.ttf"):
        try: return ImageFont.truetype(f, sz)
        except: pass
    return ImageFont.load_default()

def ask_image():
    print("\n🎯 Pick an image (JPG/PNG/WebP/BMP/TIFF ≤ 8MB) from this folder.")
    while True:
        p = input("Image path: ").strip().strip('"').strip("'")
        if not p or not os.path.isfile(p): print("⚠️ Not found."); continue
        if os.path.splitext(p)[1].lower() not in ALLOWED: print("⚠️ Unsupported type."); continue
        if os.path.getsize(p)/(1024*1024) > MAX_MB: print("⚠️ Too big (>8MB)."); continue
        try: Image.open(p).verify()
        except: print("⚠️ Corrupted image."); continue
        return p

def infer(path, img_bytes, tries=8):
    mime, _ = mimetypes.guess_type(path)
    # Prefer exact image/* content-type; fall back to multipart if guess fails
    for _ in range(tries):
        if mime and mime.startswith("image/"):
            r = requests.post(API,
                headers={"Authorization": f"Bearer {HF_API_KEY}",
                         "Content-Type": mime},
                data=img_bytes, timeout=60)
        else:
            r = requests.post(API,
                headers={"Authorization": f"Bearer {HF_API_KEY}"},
                files={"inputs": (os.path.basename(path), img_bytes, "application/octet-stream")},
                timeout=60)
        if r.status_code == 200:
            d = r.json()
            if isinstance(d, dict) and "error" in d: raise RuntimeError(d["error"])
            if not isinstance(d, list): raise RuntimeError("Bad API response.")
            return d
        if r.status_code == 503: time.sleep(2); continue
        raise RuntimeError(f"API {r.status_code}: {r.text[:300]}")
    raise RuntimeError("Model warm-up timeout.")

def draw(img, dets, thr=0.5):
    d = ImageDraw.Draw(img); f = font(18); counts = {}
    for det in dets[:50]:
        s = float(det.get("score",0));
        if s < thr: continue
        lab = det.get("label","object"); b = det.get("box",{})
        x1,y1,x2,y2 = (int(b.get(k,0)) for k in ("xmin","ymin","xmax","ymax"))
        if not (x2>0 and y2>0):
            x,y,w,h = int(b.get("x",0)), int(b.get("y",0)), int(b.get("w",0)), int(b.get("h",0))
            x1,y1,x2,y2 = x,y,x+w,y+h
        color = tuple(random.randint(80,255) for _ in range(3))
        d.rectangle([(x1,y1),(x2,y2)], outline=color, width=4)
        txt = f"{EMOJI.get(lab.lower(),'✨')} {lab} {s*100:.0f}%"
        tw,th = d.textlength(txt,font=f), f.size+6
        d.rectangle([(x1,max(0,y1-th)),(x1+tw+8,y1)], fill=color)
        d.text((x1+4,y1-th+3), txt, font=f, fill=(0,0,0))
        counts[lab] = counts.get(lab,0)+1
    return counts

def main():
    path = ask_image()
    with open(path,"rb") as fh: by = fh.read()
    try: dets = infer(path, by)
    except Exception as e: return print("❌", e)
    img = Image.open(io.BytesIO(by)).convert("RGB")
    counts = draw(img, dets, thr=0.5)
    out = f"annotated_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
    img.save(out); print(f"✅ Saved: {out}")
    if counts:
        print("🎉 I found:");
        for k,v in sorted(counts.items(), key=lambda kv:(-kv[1],kv[0])):
            print(f" • {EMOJI.get(k.lower(),'✨')} {k}: {v}")
    else:
        print("🤔 No confident detections—try a clearer or busier scene!")

    print("\n⚠️ Disclaimer: This is an AI model demo. "
          "Detections may not always be accurate or complete. "
          "Use it for fun and learning, not for safety-critical decisions.")

if __name__=="__main__": main()


In [ ]:
# Vintage Photo Healer using Stable Diffusion Inpainting
# ------------------------------------------------------
# Objective:
# Restore damaged areas of a vintage photograph using AI-driven inpainting.
# Students can repair scratches, tears, faded regions, and missing portions
# by applying masks and descriptive prompts.

# Install Required Libraries
# pip install diffusers transformers accelerate torch pillow opencv-python matplotlib

import torch
from diffusers import StableDiffusionInpaintPipeline
from PIL import Image
import matplotlib.pyplot as plt

# ------------------------------------------------------
# STEP 1: Load Stable Diffusion Inpainting Model
# ------------------------------------------------------

model_id = "runwayml/stable-diffusion-inpainting"

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)

pipe = pipe.to("cuda")   # Use GPU for faster processing

# ------------------------------------------------------
# STEP 2: Load Vintage Image and Mask
# ------------------------------------------------------
# image_path -> damaged vintage photo
# mask_path  -> white areas represent damaged regions to repair

image_path = "vintage_photo.jpg"
mask_path = "damage_mask.png"

image = Image.open(image_path).convert("RGB")
mask_image = Image.open(mask_path).convert("RGB")

# Resize for Stable Diffusion
image = image.resize((512, 512))
mask_image = mask_image.resize((512, 512))

# ------------------------------------------------------
# STEP 3: Define Restoration Prompt
# ------------------------------------------------------

prompt = """
Restore this vintage black-and-white photograph.
Repair scratches, cracks, faded areas, and torn sections.
Preserve realistic facial details, old camera texture,
historical appearance, and natural lighting.
"""

negative_prompt = """
blurry, distorted face, unrealistic colors,
modern objects, low quality, extra limbs
"""

# ------------------------------------------------------
# STEP 4: Perform AI Inpainting
# ------------------------------------------------------

result = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=image,
    mask_image=mask_image,
    guidance_scale=8.5,
    num_inference_steps=50,
    strength=0.99
).images[0]

# ------------------------------------------------------
# STEP 5: Save Restored Image
# ------------------------------------------------------

output_path = "restored_vintage_photo.png"
result.save(output_path)

print("Restoration completed!")
print(f"Saved restored image as: {output_path}")

# ------------------------------------------------------
# STEP 6: Display Results
# ------------------------------------------------------

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(image)
plt.title("Original Damaged Photo")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(mask_image)
plt.title("Damage Mask")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(result)
plt.title("Restored Photo")
plt.axis("off")

plt.show()

```python
# Object Detection (concise, correct Content-Type). Requires: pillow, requests, config.py (HF_API_KEY).
# This line is a shebang, indicating the script should be executed with python3. It's often used in executable scripts but might not be strictly necessary in a notebook context.
#!/usr/bin/env python3

# This is a comment describing the purpose of the script (Object Detection) and its dependencies.
# Object Detection (concise, correct Content-Type). Requires: pillow, requests, config.py (HF_API_KEY).

# Import necessary modules:
# os: Provides a way of using operating system dependent functionality.
# io: Provides tools for working with I/O streams.
# time: Provides various time-related functions.
# random: Generates pseudo-random numbers.
# requests: Allows sending HTTP requests.
# mimetypes: Guesses the MIME type of a file.
from datetime import datetime # Imports the datetime class from the datetime module for working with dates and times.
from PIL import Image, ImageDraw, ImageFont # Imports Image, ImageDraw, and ImageFont classes from the Pillow (PIL) library for image manipulation.
from config import HF_API_KEY # Imports HF_API_KEY from a 'config' module (presumably a local file named config.py) for API authentication.

MODEL = "facebook/detr-resnet-101" # Defines the name of the Hugging Face model to be used for object detection.
API = f"https://router.huggingface.co/hf-inference/models/{MODEL}" # Constructs the API endpoint URL using the defined MODEL.
ALLOWED, MAX_MB = {".jpg",".jpeg",".png",".bmp",".gif",".webp",".tiff"}, 8 # Defines a set of allowed image file extensions and the maximum allowed file size in megabytes.
EMOJI = {"person":"🧍","car":"🚗","truck":"🚚","bus":"🚌","bicycle":"🚲","motorcycle":"🏍️","dog":"🐶","cat":"🐱",
         "bird":"🐦","horse":"🐴","sheep":"🐑","cow":"🐮","bear":"🐻","giraffe":"🦒","zebra":"🦓","banana":"🍌",
         "apple":"🍎","orange":"🍊","pizza":"🍕","broccoli":"🥦","book":"📘","laptop":"💻","tv":"📺","bottle":"🧴","cup":"🥤"} # A dictionary mapping common object labels to their corresponding emoji for display.

def font(sz=18):
    # Defines a function to load a suitable font for drawing text on images.
    for f in ("DejaVuSans.ttf","arial.ttf"):
        # Tries to load 'DejaVuSans.ttf' or 'arial.ttf' at the specified size.
        try: return ImageFont.truetype(f, sz)
        except: pass # If a font cannot be loaded, it tries the next one.
    return ImageFont.load_default() # If neither custom font is found, it falls back to the default Pillow font.

def ask_image():
    # Defines a function to prompt the user for an image path and validate it.
    print("\n🎯 Pick an image (JPG/PNG/WebP/BMP/TIFF ≤ 8MB) from this folder.") # Instructions for the user.
    while True:
        # Loop until a valid image path is provided.
        p = input("Image path: ").strip().strip('"').strip("'") # Prompts for input and cleans the path string.
        if not p or not os.path.isfile(p): print("⚠️ Not found."); continue # Checks if the path is empty or the file doesn't exist.
        if os.path.splitext(p)[1].lower() not in ALLOWED: print("⚠️ Unsupported type."); continue # Checks if the file extension is allowed.
        if os.path.getsize(p)/(1024*1024) > MAX_MB: print("⚠️ Too big (>8MB)."); continue # Checks if the file size exceeds the limit.
        try: Image.open(p).verify()
        except: print("⚠️ Corrupted image."); continue # Attempts to open and verify the image to check for corruption.
        return p # Returns the validated image path.

def infer(path, img_bytes, tries=8):
    # Defines a function to send the image to the Hugging Face API for inference.
    mime, _ = mimetypes.guess_type(path) # Guesses the MIME type of the image based on its path.
    # Prefer exact image/* content-type; fall back to multipart if guess fails
    for _ in range(tries):
        # Loops for a number of tries, to handle potential temporary API issues (like warm-up).
        if mime and mime.startswith("image/"):
            # If a specific image MIME type is determined, send the image directly as data.
            r = requests.post(API,
                headers={"Authorization": f"Bearer {HF_API_KEY}",
                         "Content-Type": mime},
                data=img_bytes, timeout=60)
        else:
            # If the MIME type is not specific or guessed, send it as multipart form data.
            r = requests.post(API,
                headers={"Authorization": f"Bearer {HF_API_KEY}"},
                files={"inputs": (os.path.basename(path), img_bytes, "application/octet-stream")},
                timeout=60)
        if r.status_code == 200:
            # If the request was successful (HTTP 200).
            d = r.json() # Parses the JSON response.
            if isinstance(d, dict) and "error" in d: raise RuntimeError(d["error"]) # Checks for API errors in the response.
            if not isinstance(d, list): raise RuntimeError("Bad API response.") # Checks if the response is a list as expected.
            return d # Returns the detection results.
        if r.status_code == 503: time.sleep(2); continue # If service is unavailable (HTTP 503), wait and retry.
        raise RuntimeError(f"API {r.status_code}: {r.text[:300]}") # Raises an error for other HTTP status codes.
    raise RuntimeError("Model warm-up timeout.") # If all retries fail, raises a timeout error.

def draw(img, dets, thr=0.5):
    # Defines a function to draw bounding boxes and labels on the image based on detections.
    d = ImageDraw.Draw(img); f = font(18); counts = {} # Creates a Draw object, loads a font, and initializes a dictionary for object counts.
    for det in dets[:50]:
        # Iterates through up to the first 50 detections.
        s = float(det.get("score",0)); # Gets the confidence score of the detection.
        if s < thr: continue # Skips detections with a score below the threshold.
        lab = det.get("label","object"); b = det.get("box",{}) # Gets the object label and bounding box coordinates.
        x1,y1,x2,y2 = (int(b.get(k,0)) for k in ("xmin","ymin","xmax","ymax")) # Extracts xmin, ymin, xmax, ymax coordinates.
        if not (x2>0 and y2>0):
            # Handles cases where bounding box format might be x, y, width, height instead of xmin, ymin, xmax, ymax.
            x,y,w,h = int(b.get("x",0)), int(b.get("y",0)), int(b.get("w",0)), int(b.get("h",0))
            x1,y1,x2,y2 = x,y,x+w,y+h
        color = tuple(random.randint(80,255) for _ in range(3)) # Generates a random color for the bounding box.
        d.rectangle([(x1,y1),(x2,y2)], outline=color, width=4) # Draws the bounding box rectangle.
        txt = f"{EMOJI.get(lab.lower(),'✨')} {lab} {s*100:.0f}%" # Formats the label text with emoji and confidence.
        tw,th = d.textlength(txt,font=f), f.size+6 # Calculates text width and height for the background rectangle.
        d.rectangle([(x1,max(0,y1-th)),(x1+tw+8,y1)], fill=color) # Draws a background rectangle for the text.
        d.text((x1+4,y1-th+3), txt, font=f, fill=(0,0,0)) # Draws the text label.
        counts[lab] = counts.get(lab,0)+1 # Increments the count for the detected object label.
    return counts # Returns the dictionary of object counts.

def main():
    # The main function to orchestrate the object detection process.
    path = ask_image() # Calls ask_image() to get a validated image path from the user.
    with open(path,"rb") as fh: by = fh.read() # Opens the image file in binary read mode and reads its content.
    try: dets = infer(path, by) # Calls infer() to send the image for object detection.
    except Exception as e: return print("❌", e) # If inference fails, prints an error message and exits.
    img = Image.open(io.BytesIO(by)).convert("RGB") # Opens the image bytes as a Pillow Image object and converts to RGB.
    counts = draw(img, dets, thr=0.5) # Calls draw() to annotate the image with detections and get object counts.
    out = f"annotated_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png" # Generates a unique output filename for the annotated image.
    img.save(out); print(f"✅ Saved: {out}") # Saves the annotated image and prints a confirmation message.
    if counts:
        # If any objects were detected confidently.
        print("🎉 I found:");
        for k,v in sorted(counts.items(), key=lambda kv:(-kv[1],kv[0])):
            # Prints a summary of detected objects, sorted by count (descending) then by name (ascending).
            print(f" • {EMOJI.get(k.lower(),'✨')} {k}: {v}")
    else:
        # If no objects were detected confidently.
        print("🤔 No confident detections—try a clearer or busier scene!")

    print("\n⚠️ Disclaimer: This is an AI model demo. "
          "Detections may not always be accurate or complete. "
          "Use it for fun and learning, not for safety-critical decisions.") # Prints a disclaimer about the AI model.

if __name__=="__main__": main() # Ensures that main() is called only when the script is executed directly.

# The following lines are not part of the Python code itself, but are part of the Colab cell metadata.
# They indicate the name of the step (Vintage Photo Healer using Stable Diffusion Inpainting) and its objective.
# Vintage Photo Healer using Stable Diffusion Inpainting
# ------------------------------------------------------
# Objective:
# Restore damaged areas of a vintage photograph using AI-driven inpainting.
# Students can repair scratches, tears, faded regions, and missing portions
# by applying masks and descriptive prompts.

# These lines are comments explaining the next section is about installing required libraries.
# Install Required Libraries
# pip install diffusers transformers accelerate torch pillow opencv-python matplotlib

# These lines import the necessary libraries for the inpainting task.
import torch # Imports the PyTorch library, a deep learning framework.
from diffusers import StableDiffusionInpaintPipeline # Imports the StableDiffusionInpaintPipeline from the diffusers library.
from PIL import Image # Imports the Image class from Pillow for image manipulation.
import matplotlib.pyplot as plt # Imports matplotlib.pyplot for plotting and displaying images.

# These lines are comments indicating the start of STEP 1.
# ------------------------------------------------------
# STEP 1: Load Stable Diffusion Inpainting Model
# ------------------------------------------------------

model_id = "runwayml/stable-diffusion-inpainting" # Defines the ID of the pre-trained Stable Diffusion inpainting model.

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
) # Loads the pre-trained inpainting model from Hugging Face, using float16 for potentially faster and less memory-intensive computation.

pipe = pipe.to("cuda")   # Use GPU for faster processing # Moves the model to the GPU (CUDA) for accelerated processing.

# These lines are comments indicating the start of STEP 2.
# ------------------------------------------------------
# STEP 2: Load Vintage Image and Mask
# ------------------------------------------------------
# image_path -> damaged vintage photo
# mask_path  -> white areas represent damaged regions to repair

image_path = "vintage_photo.jpg" # Defines the path to the damaged vintage photo.
mask_path = "damage_mask.png" # Defines the path to the mask image, where white areas indicate damage.

image = Image.open(image_path).convert("RGB") # Opens the damaged photo and converts it to RGB format.
mask_image = Image.open(mask_path).convert("RGB") # Opens the mask image and converts it to RGB format.

# Resize for Stable Diffusion
image = image.resize((512, 512)) # Resizes the main image to 512x512 pixels, a common input size for Stable Diffusion models.
mask_image = mask_image.resize((512, 512)) # Resizes the mask image to match the main image size.

# These lines are comments indicating the start of STEP 3.
# ------------------------------------------------------
# STEP 3: Define Restoration Prompt
# ------------------------------------------------------

prompt = """
Restore this vintage black-and-white photograph.
Repair scratches, cracks, faded areas, and torn sections.
Preserve realistic facial details, old camera texture,
historical appearance, and natural lighting.
""" # Defines the positive prompt describing the desired restoration outcome.

negative_prompt = """
blurry, distorted face, unrealistic colors,
modern objects, low quality, extra limbs
""" # Defines the negative prompt, specifying things to avoid in the generated image.

# These lines are comments indicating the start of STEP 4.
# ------------------------------------------------------
# STEP 4: Perform AI Inpainting
# ------------------------------------------------------

result = pipe(
    prompt=prompt, # Passes the positive text prompt to the inpainting pipeline.
    negative_prompt=negative_prompt, # Passes the negative text prompt.
    image=image, # Provides the original damaged image.
    mask_image=mask_image, # Provides the mask indicating damaged areas.
    guidance_scale=8.5, # Sets the guidance scale, controlling how strongly the model adheres to the prompt.
    num_inference_steps=50, # Sets the number of denoising steps for image generation.
    strength=0.99 # Sets the strength of the inpainting, how much of the original image is preserved versus generated content.
).images[0] # Executes the inpainting pipeline and retrieves the first (and usually only) generated image.

# These lines are comments indicating the start of STEP 5.
# ------------------------------------------------------
# STEP 5: Save Restored Image
# ------------------------------------------------------

output_path = "restored_vintage_photo.png" # Defines the filename for the restored image.
result.save(output_path) # Saves the restored image to the specified path.

print("Restoration completed!") # Prints a message indicating completion.
print(f"Saved restored image as: {output_path}") # Prints the path where the restored image is saved.

# These lines are comments indicating the start of STEP 6.
# ------------------------------------------------------
# STEP 6: Display Results
# ------------------------------------------------------

plt.figure(figsize=(15,5)) # Creates a new matplotlib figure with a specified size.

plt.subplot(1,3,1) # Creates a subplot grid (1 row, 3 columns) and selects the first subplot.
plt.imshow(image) # Displays the original damaged image in the first subplot.
plt.title("Original Damaged Photo") # Sets the title for the first subplot.
plt.axis("off") # Turns off the axes for the subplot.

plt.subplot(1,3,2) # Selects the second subplot.
plt.imshow(mask_image) # Displays the damage mask in the second subplot.
plt.title("Damage Mask") # Sets the title for the second subplot.
plt.axis("off") # Turns off the axes for the subplot.

plt.subplot(1,3,3) # Selects the third subplot.
plt.imshow(result) # Displays the restored image in the third subplot.
plt.title("Restored Photo") # Sets the title for the third subplot.
plt.axis("off") # Turns off the axes for the subplot.

plt.show() # Displays all generated plots.
```